<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook maps where Phase 1 long-read participants were sampled relative to the broader All of Us cohort. It uses ZIP3 socioeconomic tables from Cohort Builder, places ZIP3 centroids on a US map (with Alaska and Hawaii insets), and draws a Census-division choropleth of participant counts.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
# Install cartopy (map projections) if it is missing; restart the kernel after installing.
try:
    import cartopy
except ImportError:
    print("Cartopy not found. Installing... (note, you will need to restart the Jupyter notebook kernel before proceeding).")
    !pip install cartopy

In [ ]:
import cartopy

In [ ]:
# Dataset Builder queries for the long-read cohort: demographics (this cell), survey answers, and the
# ZIP3 socioeconomic table, which gives each participant's 3-digit ZIP code.
import pandas
import os

# This query represents dataset "Long reads, basic info, socioeconomic status" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_08543380_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) )"""

dataset_08543380_person_df = pandas.read_gbq(
    dataset_08543380_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08543380_person_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Long reads, basic info, socioeconomic status" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_08543380_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1586134)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_lr_whole_genome_variant = 1 ) )
        )"""

dataset_08543380_survey_df = pandas.read_gbq(
    dataset_08543380_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08543380_survey_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Long reads, basic info, socioeconomic status" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_08543380_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) ) 
        AND observation_source_concept_id = 1585250 
        AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_08543380_zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_08543380_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08543380_zip_code_socioeconomic_df.head(5)

In [ ]:
dataset_08543380_zip_code_socioeconomic_df.zip_code

In [ ]:
# Dataset Builder queries for all participants: survey answers and ZIP3 socioeconomic table
# (the grey background points on the map).
import pandas
import os

# This query represents dataset "All participants, basic info, socioeconomic status" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_45551076_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1586134)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )"""

dataset_45551076_survey_df = pandas.read_gbq(
    dataset_45551076_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_45551076_survey_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "All participants, basic info, socioeconomic status" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_45551076_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_45551076_zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_45551076_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_45551076_zip_code_socioeconomic_df.head(5)

In [ ]:
# GeoNames US postal-code table, used to place each ZIP3 at an approximate latitude and longitude.
!wget -O US.zip https://download.geonames.org/export/zip/US.zip

In [ ]:
!unzip -o US.zip

In [ ]:
!ls

In [ ]:
!head US.txt

In [ ]:
# Load the GeoNames table.
import pandas as pd

columns = [
    'country code', 'postal code', 'place name', 'admin name1', 'admin code1', 
    'admin name2', 'admin code2', 'admin name3', 'admin code3', 
    'latitude', 'longitude', 'accuracy'
]

# Read the file into a Pandas DataFrame
df = pd.read_csv('US.txt', sep='\t', header=None, names=columns)

In [ ]:
df

In [ ]:
# Collapse to one representative location per 3-digit ZIP prefix.
df['postal code'] = df['postal code'].astype(str)

# Trim the postal code to the first three digits
df['postal code'] = df['postal code'].str[:3]

# Group by the modified postal code and take one row from each group
df_grouped = df.groupby('postal code').first().reset_index()

In [ ]:
# Map of the continental US: grey points are ZIP3s of all participants, red points are ZIP3s of
# long-read participants.
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Extract unique 3-digit ZIP codes from the DataFrame
all_zip3 = dataset_45551076_zip_code_socioeconomic_df['zip_code'].str[:3].unique()
highlight_zip3 = dataset_08543380_zip_code_socioeconomic_df['zip_code'].str[:3].unique()

# Example centroids or approximate locations for 3-digit ZIP codes
zip3_locations = df_grouped.set_index('postal code')[['longitude', 'latitude']].apply(tuple, axis=1).to_dict()

# Filter locations to include only the ZIPs in your DataFrame
all_filtered_locations = {zip3: coords for zip3, coords in zip3_locations.items() if zip3 in all_zip3}
highlight_filtered_locations = {zip3: coords for zip3, coords in zip3_locations.items() if zip3 in highlight_zip3}

# Create the map
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': ccrs.PlateCarree()})
#ax.set_extent([-128, -65, 20, 49])  # Continental US bounds
ax.set_extent([-125, -67, 24, 47])  # Continental US bounds

# Add state boundaries and coastlines
#ax.add_feature(cfeature.LAND, edgecolor='black', zorder=0, facecolor='white')  # Add land background
ax.add_feature(cfeature.LAND, edgecolor='black', zorder=0, facecolor='white')  # Add land background
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', zorder=0)  # Add ocean background
ax.add_feature(cfeature.STATES, edgecolor='grey', linewidth=0.5, zorder=1)  # Add state boundaries
ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.8, zorder=1)  # Add coastlines
ax.add_feature(cfeature.BORDERS, edgecolor='black', linewidth=0.8, zorder=1)  # Add country borders
ax.add_feature(cfeature.LAKES, facecolor='lightblue', zorder=0)
#ax.add_feature(cfeature.RIVERS, facecolor='lightblue', zorder=0)

# Highlight ZIP code regions
for zip3, (lon, lat) in all_filtered_locations.items():
    ax.plot(lon, lat, 'o', color='#cccccc77', markeredgecolor='dimgray', markeredgewidth=0.5, markersize=4, label=f"ZIP {zip3}")  # Red dots for ZIP code locations
    
for zip3, (lon, lat) in highlight_filtered_locations.items():
    ax.plot(lon, lat, 'o', color='#ff0000aa', markeredgecolor='dimgray', markeredgewidth=0.5, markersize=4, label=f"ZIP {zip3}")  # Red dots for ZIP code locations

# Save the map as a PDF
plt.savefig("highlighted_zip_codes_map.pdf", format="pdf", bbox_inches="tight")
    
# Show the map
plt.show()

In [ ]:
# Same map with Alaska and Hawaii insets.
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Create main figure and axis
fig = plt.figure(figsize=(10, 7))
main_ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
main_ax.set_extent([-128, -65, 24, 50], crs=ccrs.PlateCarree())  # Mainland US

# Base map features
for ax in [main_ax]:
    ax.add_feature(cfeature.LAND, facecolor='white', edgecolor='black')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.STATES, edgecolor='gray', linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, edgecolor='black')
    ax.add_feature(cfeature.LAKES, facecolor='lightblue')

# -- Add Alaska inset --
ax_ak = fig.add_axes([0.03, 0.12, 0.25, 0.25], projection=ccrs.PlateCarree())  # [left, bottom, width, height]
ax_ak.set_extent([-170, -130, 52, 72], crs=ccrs.PlateCarree())
ax_ak.add_feature(cfeature.LAND, facecolor='white', edgecolor='black')
ax_ak.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax_ak.add_feature(cfeature.STATES, edgecolor='gray', linewidth=0.5)
ax_ak.add_feature(cfeature.BORDERS, edgecolor='black')
ax_ak.add_feature(cfeature.LAKES, facecolor='lightblue')

# -- Add Hawaii inset --
ax_hi = fig.add_axes([0.72, 0.08, 0.18, 0.18], projection=ccrs.PlateCarree())
ax_hi.set_extent([-161, -154, 18, 23], crs=ccrs.PlateCarree())
ax_hi.add_feature(cfeature.LAND, facecolor='white', edgecolor='black')
ax_hi.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax_hi.add_feature(cfeature.STATES, edgecolor='gray', linewidth=0.5)
ax_hi.add_feature(cfeature.BORDERS, edgecolor='black')
ax_hi.add_feature(cfeature.LAKES, facecolor='lightblue')

# -- Plot points for each region --
def plot_points(ax, points_dict, color):
    for zip3, (lon, lat) in points_dict.items():
        ax.plot(lon, lat, 'o', color=color, markersize=3, markeredgecolor='dimgray', markeredgewidth=0.5)

# Plot all points
plot_points(main_ax, all_filtered_locations, '#cccccc77')
plot_points(main_ax, highlight_filtered_locations, '#ff0000aa')

plot_points(ax_ak, all_filtered_locations, '#cccccc77')
plot_points(ax_ak, highlight_filtered_locations, '#ff0000aa')

plot_points(ax_hi, all_filtered_locations, '#cccccc77')
plot_points(ax_hi, highlight_filtered_locations, '#ff0000aa')

# Save or show
plt.savefig("highlighted_zip_codes_map.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
dataset_08543380_zip_code_socioeconomic_df

In [ ]:
# pgeocode supplies a ZIP5-to-state table.
!pip install pgeocode uszipcode

In [ ]:
# Count long-read participants per state: map each ZIP3 to its most common state.
import pandas as pd
import pgeocode

# --- Your Phase 1 dataframe ---
df = dataset_08543380_zip_code_socioeconomic_df.copy()

# Extract ZIP3 from masked ZIP codes (e.g. '112**' -> '112')
df["zip3"] = df["zip_code"].astype(str).str.extract(r"(\d{3})")

# --- Build ZIP3 -> state crosswalk vectorized ---
nomi = pgeocode.Nominatim("us")
all_zip5 = nomi._data[["postal_code", "state_code"]].dropna().copy()
all_zip5["zip5"] = all_zip5["postal_code"].astype(str).str.zfill(5)
all_zip5["zip3"] = all_zip5["zip5"].str[:3]

zip3_to_state = (
    all_zip5.groupby("zip3")["state_code"]
            .agg(lambda s: s.mode().iat[0])   # modal state per prefix
            .to_dict()
)

# --- Map ZIP3 to state in your data ---
df["state"] = df["zip3"].map(zip3_to_state)

# --- Count participants by state ---
state_counts = df.groupby("state").size().reset_index(name="count")
state_counts

In [ ]:
# Census-division choropleth of participant counts, bucketed, with each division's share labelled.
# It uses division_geoms, division_to_count, mpl (matplotlib), and pe (matplotlib.patheffects), which
# are defined in steps 1-3 of the original notebook; those cells are not in this copy.
# ----------------------------
# 4) Plot — discrete buckets (Greens), crisp borders, labeled divisions
# ----------------------------

import matplotlib.patches as mpatches

# Choose bucket edges & labels (edit to taste)
bins   = [-0.1, 0, 9, 49, 199, 9999]     # 0, 1–9, 10–49, 50–199, 200+
labels = ["0", "1–9", "10–49", "50–199", "200+"]

# Discrete colormap with as many shades as buckets
cmap = mpl.cm.get_cmap("Greens", len(labels))

# Helper: find the bin index for a value
def bin_index(v):
    for i in range(len(bins) - 1):
        if bins[i] < v <= bins[i + 1]:
            return i
    return len(labels) - 1  # fallback to last bucket

# Map each division to a facecolor from the discrete palette
div_to_color = {}
for dv in division_geoms:
    cnt = division_to_count.get(dv, 0)
    if cnt <= 0:
        div_to_color[dv] = (0.93, 0.93, 0.93, 1.0)  # light grey for zero
    else:
        div_to_color[dv] = cmap(bin_index(cnt))

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([-125, -67, 24, 47])  # CONUS

# Base layers
ax.add_feature(cfeature.LAND,    edgecolor='black', zorder=0, facecolor='white')
ax.add_feature(cfeature.OCEAN,   facecolor='lightblue', zorder=0)
ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.8, zorder=1)
ax.add_feature(cfeature.BORDERS,   edgecolor='black', linewidth=0.8, zorder=1)
ax.add_feature(cfeature.LAKES,     facecolor='lightblue', zorder=0)

# Draw divisions with discrete face colors
for dv, geom in division_geoms.items():
    if geom is None:
        continue
    ax.add_geometries([geom], ccrs.PlateCarree(),
                      facecolor=div_to_color.get(dv, (0.93, 0.93, 0.93, 1.0)),
                      edgecolor='black', linewidth=0.8, zorder=2)

# Overlay **crisper** state borders
ax.add_feature(cfeature.STATES, edgecolor="0.35", facecolor="none", linewidth=0.6, zorder=3)

# --- percentages per division ---
total_div = sum(division_to_count.values()) or 1  # avoid div-by-zero
div_to_pct = {dv: (division_to_count.get(dv, 0) / total_div) * 100.0
              for dv in division_geoms}

# Optional manual nudges to reduce overlap (tweak as needed)
offsets = {
    "West North Central": (-1.5,  0.5),
    "East North Central": ( 1.0, -0.5),
    "Middle Atlantic":    ( 1.0,  0.5),
}

# Label divisions + percentage beneath
for dv, geom in division_geoms.items():
    if geom is None:
        continue
    x, y = list(geom.representative_point().coords)[0]
    dx, dy = offsets.get(dv, (0, 0))

    # Division name
    ax.text(x+dx, y+dy, dv,
            transform=ccrs.PlateCarree(),
            ha="center", va="center",
            fontsize=7, fontweight="bold", color="black",
            path_effects=[pe.withStroke(linewidth=2, foreground="white")],
            zorder=4)

    # Percentage on a second line just below (slightly smaller)
    ax.text(x+dx, y+dy-1.0, f"{div_to_pct.get(dv, 0):.1f}%",
            transform=ccrs.PlateCarree(),
            ha="center", va="center",
            fontsize=6, fontweight="bold", color="black",
            path_effects=[pe.withStroke(linewidth=2, foreground="white")],
            zorder=4)

# Build legend handles: one for "0", the rest for bins >0
handles = []
handles.append(mpatches.Patch(facecolor=(0.93, 0.93, 0.93, 1.0),
                              edgecolor='dimgray', label="0"))

# Only add patches for >0 bins
for i, lab in enumerate(labels[1:], start=1):
    handles.append(mpatches.Patch(facecolor=cmap(i), edgecolor='dimgray', label=lab))

leg = ax.legend(
    handles=handles,
    title="Participants",
    loc="lower right",
    frameon=True,
    framealpha=0.9,
    fontsize=8,
    title_fontsize=9,
    handlelength=1.5,  # shorter color patches
    handletextpad=1.0, # tighter gap between patch and text
    borderpad=0.8,     # less empty space inside legend box
    labelspacing=0.4,  # tighter vertical spacing
)
for text in leg.get_texts():
    text.set_horizontalalignment("left")

# Save
plt.savefig("phase1_division_choropleth_buckets.pdf", format="pdf", bbox_inches="tight")
plt.show()